### Loss Functions & Regularization

- [Loss Functions](#Loss-Functions)
- [Evalution Metrics](#evaluation-metrics--imbalanced-data)
- [Regularization](#Regularization)

#### Loss Functions

1. Mean Absolute Error (MAE) / $L_1$ Loss- Calculates the average absolute magnitude of errors between predicted ($\hat{y}_i$) and actual ($y_i$) values. $\text{MAE} = \frac{1}{N} \sum_{i=1}^N \vert{}y_i - \hat{y}_i\vert{}$  Treats all error magnitudes linearly. Derivative is non-differentiable at $y_i = \hat{y}_i$ (gradient is fixed at $\pm 1$), causing instability near minimum during gradient descent. Invariant to error scale — a $10 error on a $100 item equals a $10 error on a $1,000,000 item.
    - Real-World Scenario: E-Commerce Delivery Time Prediction, Sensor Data Analysis.Why it fits: Real-world logistics data often contains extreme outliers (e.g., a package delayed by 15 days due to a snowstorm). MSE would let that single delayed package corrupt the entire model; MAE provides a linear penalty, focusing the model on median delivery performance.

2. Mean Squared Error (MSE) / $L_2$ Loss - Calculates the average of the squared differences between predictions and actual targets.  Quadratic penalty—larger errors incur exponentially higher losses than smaller errors. Highly sensitive to outliers. Errors are in squared units (e.g., $\text{USD}^2$), (use RMSE = $\sqrt{\text{MSE}}$ for target unit alignment). $\text{MSE} = \frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2$
    

3. Mean Absolute Percentage Error (MAPE) - Measures average relative percentage error relative to actual target scale. Expresses error as a percentage of actual values, making it scale-independent across varied target magnitudes. $\text{MAPE} = \frac{100\%}{N} \sum_{i=1}^N \left\vert{} \frac{y_i - \hat{y}_i}{y_i} \right\vert{}$
    - Real-World Scenario: Retail Inventory & Sales Forecasting Across Store Networks. Why it fits: Forecasts span vastly different scales—a store selling 10,000 milk cartons vs. 5 specialized electronics per week. An absolute error of 10 units is negligible for milk but devastating for laptops. MAPE evaluates error as a relative percentage, enabling cross-product benchmarking.

4. Huber Loss (Smooth $L_1$ Loss)Combines MSE and MAE via threshold parameter $\delta$.Formula:$$L_\delta(y, \hat{y}) = \begin{cases} \frac{1}{2}(y - \hat{y})^2 & \text{for } \vert{}y - \hat{y}\vert{} \le \delta \\ \delta \vert{}y - \hat{y}\vert{} - \frac{1}{2}\delta^2 & \text{otherwise} \end{cases}$$How it Works: Behaves like MSE for small errors ($\vert{}e\vert{} \le \delta$) for smooth convergence, and like MAE for large errors ($\vert{}e\vert{} > \delta$) _to limit outlier influence._
    - Real-World Scenario: Autonomous Vehicle Trajectory Steering, Robust Financial Trading Models. Why it fits: Combines the best of both worlds. For minor trajectory corrections ($\vert{}e\vert{} \le \delta$), it uses MSE for differentiable, smooth control. If a sensor glitch causes a massive jump ($\vert{}e\vert{} > \delta$), it switches to MAE linear bounds to prevent steering wild swings.

5. Entropy - Information = surprise. Rain in a rainforest carries ~0 information (expected, not surprising). Rain in the Sahara carries a lot of information (very surprising). Entropy $H(P)$ is simply the expected value (average) of information/surprise across all possible outcomes of a probability distribution $P$: $$H(P) = \mathbb{E}_{x \sim P}[I(x)] = \sum_{x} P(x) \cdot I(x)$$ $$H(P) = -\sum_{x} P(x) \log(P(x))$$

6. Cross-entropy - Suppose the true data distribution is $P$, but our model predicts a distribution $Q$. Cross-Entropy $H(P, Q)$ measures the average surprise when we observe events from true distribution $P$, but measure them using our predicted model probabilities $Q$:$$H(P, Q) = \mathbb{E}_{x \sim P}[I_Q(x)] = -\sum_{x} P(x) \log(Q(x))$$

7. Binary Cross-Entropy (BCE)In binary classification, an outcome $x$ can only be $1$ or $0$ :True distribution $P$ : $P(x=1) = y$, $P(x=0) = 1 - y$ Predicted distribution $Q$ : $Q(x=1) = p$, $Q(x=0) = 1 - p$ Plugging $P$ and $Q$ into the cross-entropy formula: $$H(P, Q) = -\left[ P(x=1)\log(Q(x=1)) + P(x=0)\log(Q(x=0)) \right]$$ $$\mathcal{L}_{\text{BCE}} = -\left[ y \log(p) + (1 - y) \log(1 - p) \right]$$ Averaging across $N$ training samples yields the standard Binary Cross-Entropy loss:$$\mathcal{L}_{\text{BCE}} = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \log(p_i) + (1 - y_i) \log(1 - p_i) \right]$$

8. Categorical Cross-Entropy (Multi-Class) Extends Log Loss to $K > 2$ mutually exclusive classes using a Softmax activation output ($\sum p_k = 1$).Formula: $$\mathcal{L}_{\text{CCE}} = -\frac{1}{N} \sum_{i=1}^N \sum_{k=1}^K y_{i,k} \log(p_{i,k})$$ (where $y_{i,k}$ is 1 if class $k$ is the ground truth, else 0) Measures the distance between the true one-hot encoded vector and predicted probability distribution across all classes. Single-label multi-class problems (e.g., Digit recognition 0–9, ImageNet classification).

    - Sparse Categorical Cross-Entropy - Mathematically identical to Categorical Cross-Entropy, but optimizes memory usage during computation.How it Works: Accepts target labels as simple integer indices ($y = 2$) rather than memory-heavy one-hot encoded vectors ($y = [0, 0, 1, 0]$).Usage: High-cardinality multi-class classification (e.g., predicting 10,000+ vocabulary words in language models).

9. Focal Loss- Modifies Binary Cross-Entropy by adding a modulating factor $(1 - p_t)^\gamma$ to handle extreme class imbalance. Formula: $$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$ (where $p_t$ is probability of the correct class, $\alpha_t$ balances class weight, $\gamma$ is focusing parameter) How it Works: Dynamically suppresses loss contributions from well-classified "easy" examples ($p_t \to 1$), forcing the model to focus gradient updates on rare, hard minority samples.Usage: Object detection (e.g., RetinaNet), rare medical anomaly detection, fraud detection.


10.  Hinge Loss (SVM Loss)A margin-based loss function designed for Support Vector Machines (SVMs) with class labels $y \in \{-1, +1\}$.Formula: $$\mathcal{L}_{\text{Hinge}} = \max(0, 1 - y \cdot f(x))$$ How it Works: Incurs a penalty if a sample is misclassified or lies within the decision boundary margin ($y \cdot f(x) < 1$). Outputs zero loss once a point is correctly classified outside the margin.Usage: Maximum-margin classifiers, linear SVMs, text classification.

#### Evaluation Metrics & Imbalanced Data

**1. Confusion Matrix**

A 2x2 (binary) or KxK (multiclass) grid: rows = actual class, columns = predicted class.

Setup: spam classifier, 100 test emails — 20 actually spam, 80 actually not-spam. Model predicts 18 as spam, 82 as not-spam.

1. The four cells (binary case), "positive" = spam:
   - TP (True Positive): predicted spam, actually spam
   - TN (True Negative): predicted not-spam, actually not-spam
   - FP (False Positive) = Type I error: predicted spam, actually not-spam
   - FN (False Negative) = Type II error: predicted not-spam, actually spam

```
               predicted spam   predicted not-spam
   actual spam        15 (TP)          5 (FN)
   actual not-spam     3 (FP)         77 (TN)
```
2. Precision = TP/(TP+FP) — of everything predicted positive, how much really is. Recall = TP/(TP+FN) — of everything actually positive, how much did we catch . F1 = 2·(P·R)/(P+R) — harmonic mean, penalizes a big P/R gap more than a plain average would

3. F-beta = (1+β²)·(P·R)/(β²·P+R) — generalizes F1; β>1 weights recall more, β<1 weights precision more. F1 is the special case β=1. Worked (β=2, recall-weighted — e.g. fraud, where catching cases matters more than false alarms): F2 = 5·(0.833×0.75)/(4×0.833+0.75) = 3.125/4.083 ≈ 0.766 (pulled lower than F1 toward recall=0.75, since β=2 cares less about precision's 0.833)

4. Specificity (TNR) = TN/(TN+FP) — of all actual negatives, how many correctly identified as negative (recall's mirror image, for the negative class) Worked (spam example): 77/(77+3) = 0.9625

```
New setup : for macro/micro/weighted — 3-class email routing, UNBALANCED on purpose (spam=5, promotions=5, primary=20 support, 30 total), so macro and weighted actually diverge this time:
    - spam:       TP=1, FN=4, FP=1 → precision=0.5, recall=0.2, F1≈0.286
    - promotions: TP=4, FN=1, FP=0 → precision=1.0, recall=0.8, F1≈0.889
    - primary:    TP=19, FN=1, FP=5 → precision≈0.792, recall=0.95, F1≈0.864
```
5. Macro avg F1 = (0.286+0.889+0.864)/3 ≈ 0.680 — treats spam's bad score (rare class, only 5 support) as equally important as primary's good one

6. Weighted avg F1 = (5×0.286 + 5×0.889 + 20×0.864)/30 ≈ 0.772 — pulled higher because primary (20/30 of the data) dominates the average, spam's ailure gets diluted to a fraction of its macro-avg impact

7. Micro avg: pool TP/FP/FN across ALL classes first, then compute one global precision/recall/F1, not an average of per-class scores.
    - ΣTP=1+4+19=24, ΣFP=1+0+5=6, ΣFN=4+1+1=6
    - micro_precision = 24/(24+6) = 0.8, micro_recall = 24/(24+6) = 0.8, micro_F1 = 0.8
    - Note: in single-label multiclass, ΣFP always equals ΣFN (every wrong prediction is simultaneously one FP for the predicted class and one FN for the true class) — s micro_precision = micro_recall = micro_F1 = accuracy exactly. Confirm: accuracy = 24/30 = 0.8

**2. Imabalanced Data**

1. SMOTE (Synthetic Minority Over-sampling Technique) -  Creates artificial minority samples along line segments connecting $k$-nearest minority neighbors. Pick a minority point $\mathbf{x}_i$. Find its $k$-nearest neighbors among minority samples; pick one randomly, $\mathbf{x}_{\text{zi}}$. Generate synthetic sample:$$\mathbf{x}_{\text{new}} = \mathbf{x}_i + \lambda (\mathbf{x}_{\text{zi}} - \mathbf{x}_i) \quad \text{where } \lambda \sim U(0, 1)$$

> Traps & Pitfalls: Overfitting / Data Leakage:  Always run SMOTE only on training folds. Noise Propagation: If minority points are noise/outliers, SMOTE creates synthetic points deep inside majority class clusters.

2. Focal Loss - Adds a modulating factor $(1 - p_t)^\gamma$ to cross-entropy loss to dynamically down-weight easy examples during training:$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$(where $p_t$ is predicted probability for ground truth class, $\alpha_t$ balances class frequency, $\gamma \ge 0$ controls focusing rate)Traps & Pitfalls: Sensitive to the hyperparameter $\gamma$. If set too high, model gradients are dominated entirely by extreme outliers

3. Precision, Recall & $F_\beta$- ScoreFormulas:$$\text{Precision} = \frac{TP}{TP + FP}, \quad \text{Recall} = \frac{TP}{TP + FN}$$ 
$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \times \text{Recall}}{(\beta^2 \cdot \text{Precision}) + \text{Recall}}$$

$\beta = 1$: $F_1$-score (harmonic mean).$\beta = 2$: Weights Recall twice as heavily as Precision (ideal for medical diagnostics/fraud detection).

4. PR-AUC vs. ROC-AUC - Formulas & Logic:ROC Curve: Plots True Positive Rate ($\frac{TP}{TP+FN}$) vs. False Positive Rate ($\frac{FP}{FP+TN}$).PR Curve: Plots Precision vs. Recall. Traps & Pitfalls (ROC-AUC Trap): On datasets with extreme imbalance (e.g., 99.9% negative), the False Positive Rate denominator ($FP + TN$) is dominated by $TN$. A massive rise in absolute false positives barely changes FPR, giving a misleadingly high ROC-AUC ($>0.95$) despite poor real-world model performance. Always use PR-AUC for severe class imbalance.

#### Regularization

1. $L_1$ Regularization (Lasso)Adds the sum of absolute values of weights as a penalty term to the loss function.Formula: $$\mathcal{L}_{\text{Lasso}} = \mathcal{L}_{\text{Base}} + \lambda \sum_{j=1}^p \vert{}w_j\vert{}$$  Drives less important feature weights strictly to zero, performing built-in feature selection. High-dimensional, sparse datasets where many features are irrelevant.

2. $L_2$ Regularization (Ridge)Adds the sum of squared weights to the loss function.Formula: $$\mathcal{L}_{\text{Ridge}} = \mathcal{L}_{\text{Base}} + \lambda \sum_{j=1}^p w_j^2$$ . Shrinks weight values uniformly toward zero, but never makes them exactly zero. Best Used For: Handling multicollinearity (correlated features) and stabilizing models.

3. ElasticNet Combines both $L_1$ and $L_2$ penalties with a blending ratio $r \in [0, 1]$.Formula: $$\mathcal{L}_{\text{ElasticNet}} = \mathcal{L}_{\text{Base}} + \lambda \left( r \sum_{j=1}^p \vert{}w_j\vert{} + \frac{1 - r}{2} \sum_{j=1}^p w_j^2 \right)$$ Best Used For: Datasets with groups of correlated features (Lasso arbitrarily picks one; ElasticNet retains the whole group).

4. Neural Network Specific Regularization 
    1. Dropout: Randomly deactivates a fraction $p$ of neurons during each training step. Prevents co-adaptation of features and acts like ensembling many smaller networks.
    2. Early Stopping: Monitors validation loss during training and halts training when validation loss starts increasing (before overfitting sets in).
    3. Batch Normalization: Normalizes layer inputs ($\mu=0, \sigma=1$) per batch. Primarily stabilizes training, but adds mild stochastic noise that acts as a regularizer.
    5. Weight Decay: The explicit gradient update formulation of $L_2$ regularization in deep learning optimizers (e.g., AdamW).

5. Tree-Based Regularization (Decision Trees, XGBoost, Random Forest) - Pre-Pruning Parameters: `max_depth`: Limits tree depth.  `min_samples_split` / `min_child_weight`: Minimum samples needed to split an internal node `min_samples_leaf`: Minimum samples required at a leaf node. Subsampling & Colsample: Randomly sampling rows (`subsample`) and feature columns (`colsample_bytree`) per split/tree (used in XGBoost and Random Forest).